# 06 · Os 5 erros reais

Fase 8 do plano. Na conferência manual do test, o C2 teve **13 falhas** nas perguntas respondíveis
(respostas parciais, incorretas ou recusadas). Cada uma recebeu uma categoria da taxonomia da
seção 7.4 do plano (`avaliacao/revisao/erros_test.yaml`). O Pedro escolheu 5 representativas:
**T-021, T-028, T-007, T-039 e T-011**.

Para cada erro, o notebook mostra a entrada, a saída, a causa-raiz, a correção proposta e o
efeito medido depois. As correções foram medidas no mesmo test, em duas versões do C2:
- **C2+A+C:** reescrita da pergunta (A) e busca híbrida com reranker (C);
- **C2+M:** multiconsulta, o sistema oficial (3 reformulações, fusão RRF, reranker e 5 trechos).

O test já tinha sido visto quando as correções foram feitas, então o efeito aqui é **pós-correção**;
a medida limpa da M está no T3 (notebook 05, seção 7).

Os trechos de páginas gov.br aparecem como hash (licença CC BY-ND); o texto completo fica só nas
cópias locais.

In [1]:
import json
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
import yaml
from IPython.display import Markdown, display

from src.avaliacao.casos import carregar_congelado

A = RAIZ / "avaliacao"
REV = A / "revisao"
Q = "qwen3_4b-instruct-2507-q4_K_M"
pd.set_option("display.max_colwidth", 200)


def ler_yaml(nome):
    return yaml.safe_load((REV / nome).read_text(encoding="utf-8"))


def run(sufixo):
    with open(A / "runs" / "geracao" / f"C2_test_{Q}{sufixo}.jsonl", encoding="utf-8") as f:
        return {(x := json.loads(l))["id"]: x for l in f if l.strip()}


casos = {c.id: c for c in carregar_congelado("test")}
erros = ler_yaml("erros_test.yaml")["falhas"]
conf_c2 = ler_yaml("conferencia_test.yaml")["itens"]["C2"]
conf_ac = ler_yaml("conferencia_pos_correcao.yaml")["itens"]["C2_AC"]
herd_ac = ler_yaml("conferencia_pos_pre.yaml")["herdadas"]
conf_m = ler_yaml("conferencia_m.yaml")["test"]
execucoes = {"C2 (original)": run(""), "C2+A+C": run("_corrAC"), "C2+M (oficial)": run("_corrM")}


def veredito(nome, id_):
    """Veredito manual, com as heranças registradas em cada conferência."""
    r = execucoes[nome][id_]["saida"]["resposta"]
    if r is None or r["recusa"]:
        return {"veredito": "recusa", "falso": "nao"}
    if nome == "C2+A+C":
        return conf_ac.get(id_) or conf_c2[id_]
    if nome == "C2+M (oficial)":
        return conf_m["itens"].get(id_) or conf_c2[id_]
    return conf_c2[id_]


CATEGORIAS = {1: "recuperação", 2: "fragmentação", 3: "recusa indevida", 5: "número alterado",
              6: "citação não sustenta", 8: "regra obsoleta"}


def mostrar(id_):
    c = casos[id_]
    e = erros[id_]
    display(Markdown(f"**Pergunta ({c.tipo}):** {c.pergunta}\n\n**Gabarito:** {c.resposta_gabarito}\n\n"
                     f"**Categoria:** {CATEGORIAS[e['categoria']]}"
                     + (f" (secundária: {CATEGORIAS[e['secundaria']]})" if e["secundaria"] else "")
                     + f" · gold entre os 3 trechos: {e['gold_no_top3']} · nota do portão: {e['nota']}"))
    linhas = []
    for nome, ex in execucoes.items():
        s = ex[id_]["saida"]
        v = veredito(nome, id_)
        fontes = "; ".join(f"{f['id']}: {f['trecho_citado'][:110]}" for f in s["resposta"]["fontes"])
        linhas.append({"configuração": nome, "resposta": s["resposta"]["resposta"],
                       "veredito": v["veredito"], "afirma algo falso": v["falso"],
                       "trechos recebidos": ", ".join(t["chunk_id"] for t in s["trechos"]),
                       "fontes citadas": fontes})
    return pd.DataFrame(linhas).set_index("configuração")

## As 13 falhas do C2 no test e as 5 escolhidas

As 5 cobrem as quatro causas mais frequentes: número de outra situação (T-021), trecho certo
interpretado ao contrário (T-028), regra revogada (T-007), resposta que precisava de dois
trechos (T-039) e exceção apresentada como regra (T-011).

In [2]:
ESCOLHIDOS = ["T-021", "T-028", "T-007", "T-039", "T-011"]
tab = pd.DataFrame(erros).T
tab["categoria"] = tab.categoria.map(CATEGORIAS)
tab["secundaria"] = tab.secundaria.map(CATEGORIAS)
tab["escolhido"] = ["✓" if i in ESCOLHIDOS else "" for i in tab.index]
tab[["escolhido", "veredito", "categoria", "secundaria", "gold_no_top3", "nota"]]

,escolhido,veredito,categoria,secundaria,gold_no_top3,nota
T-003,,parcial,citação não sustenta,número alterado,sim,0.7476
T-007,✓,recusa,recusa indevida,regra obsoleta,sim,0.6184
T-011,✓,incorreta,recuperação,citação não sustenta,nao,0.7085
T-021,✓,parcial,número alterado,citação não sustenta,nao,0.7215
T-024,,recusa,recusa indevida,recuperação,nao,0.6017
T-028,✓,incorreta,citação não sustenta,NaN,sim,0.7021
T-032,,recusa,recusa indevida,NaN,sim,0.5895
T-033,,parcial,recuperação,número alterado,nao,0.6494
T-034,,recusa,recusa indevida,recuperação,nao,0.5954
T-036,,recusa,recusa indevida,recuperação,nao,0.5426


## Erro 1 · T-021: o prazo de outra situação

**Causa-raiz.** O trecho gold (art. 360 da REN 1.000: aviso de 15 dias no corte por falta de
pagamento) não veio na busca. Veio a página "Como resolver" da ANEEL sobre corte de energia
(D14-q10), que traz na mesma frase dois prazos: 3 dias para corte por razões técnicas ou de
segurança e 15 dias para falta de pagamento. O modelo pegou o número da situação errada. A
verificação automática de números não detecta esse erro, porque "três dias" está no trecho.

**Correção proposta.**
1. Um passo que identifica a situação do consumidor (aqui, falta de pagamento) antes de escolher
   o prazo.
2. Dividir as páginas explicativas que juntam várias situações em trechos por situação.
3. Dar preferência aos atos normativos sobre as páginas explicativas (correção E, testada só no
   dev e no T2).

**Efeito medido.** Nenhuma das versões corrigiu: o art. 360 não veio nem com o reranker nem com
os 5 trechos da multiconsulta, e as duas continuam dizendo "três dias". As propostas 1 e 2 não
foram implementadas.

In [3]:
mostrar("T-021")

**Pergunta (coloquial):** Chegou um papel dizendo que vão cortar minha luz por causa de conta atrasada. Eles podem cortar sem me avisar antes? Quantos dias de aviso eu tenho?

**Gabarito:** Não. No caso de inadimplemento, a notificação deve ser feita com pelo menos 15 dias de antecedência.

**Categoria:** número alterado (secundária: citação não sustenta) · gold entre os 3 trechos: nao · nota do portão: 0.7215

,resposta,veredito,afirma algo falso,trechos recebidos,fontes citadas
configuração,,,,,
C2 (original),"Não, eles não podem cortar sua luz sem avisar. A distribuidora precisa avisar com no mínimo três dias de antecedência, se for por falta de pagamento. Se você quitar a dívida no momento do corte, a...",parcial,sim,"D14-q11, D14-q10, D07-s8.2-item198",D14-q10: [omitido: página gov.br (CC BY-ND 3.0); sha256 fa7fbcaeba38; 278 caracteres]
C2+A+C,"Não, eles não podem cortar sua luz sem avisar antes. A distribuidora precisa avisar por escrito com no mínimo três dias de antecedência, nos casos de falta de pagamento. Se você apresentar comprov...",parcial,sim,"D14-q10, D14-q11, D01-art436-p1",D14-q10: [omitido: página gov.br (CC BY-ND 3.0); sha256 5bc1eb2acc95; 400 caracteres]; D14-q10: [omitido: página gov.br (CC BY-ND 3.0); sha256 e5e754273931; 326 caracteres]
C2+M (oficial),"Não, eles não podem cortar sem avisar. A distribuidora precisa avisar pelo menos três dias antes, por escrito ou impresso, na fatura, por falta de pagamento.",parcial,sim,"D14-q10, D14-q11, D14-q14, D07-s8.2-item198, D01-art436-p1",D14-q10: [omitido: página gov.br (CC BY-ND 3.0); sha256 fa7fbcaeba38; 278 caracteres]


## Erro 2 · T-028: o trecho certo, lido ao contrário

**Causa-raiz.** O gold (PRORET, submódulo 6.8, item 37: nas bandeiras amarela e vermelha há um
adicional, destacado na fatura) veio em 3º lugar. O modelo seguiu a página de perguntas frequentes
da ANEEL sobre bandeiras, que apresenta a bandeira como uma forma de mostrar um custo que já
existia, e respondeu "não é uma taxa a mais". O sentido ficou invertido.

**Correção proposta.** Quando o ato normativo e a página explicativa parecem divergir, responder
pelo ato normativo (correção E) e dizer o "sim" ou o "não" com base nele.

**Efeito medido.** Melhorou em parte. O C2+A+C passou a reconhecer uma cobrança adicional
proporcional ao consumo. O C2+M já diz que o valor extra vem destacado na fatura, mas continua
começando com "Não": a resposta se contradiz (parcial, afirma algo falso).

In [4]:
mostrar("T-028")

**Pergunta (coloquial):** Minha conta veio com bandeira vermelha. Isso é uma taxa a mais que vão me cobrar?

**Gabarito:** Sim: nas bandeiras amarela e vermelha há um adicional, que deve vir destacado na fatura; na bandeira verde não há acréscimo.

**Categoria:** citação não sustenta · gold entre os 3 trechos: sim · nota do portão: 0.7021

,resposta,veredito,afirma algo falso,trechos recebidos,fontes citadas
configuração,,,,,
C2 (original),"Não. A bandeira vermelha não é uma taxa a mais. É apenas uma forma de mostrar o custo real da energia no momento em que você consome. O valor já estava na sua conta, mas agora está mais claro.",incorreta,sim,"D15-q02, D15-q05, D12-s6-item37",D15-q02: [omitido: página gov.br (CC BY-ND 3.0); sha256 4ab00ebaa6f5; 400 caracteres]
C2+A+C,"Não. A bandeira vermelha não é uma taxa a mais que você paga de forma separada. Ela indica que os custos de geração da energia estão mais altos, e por isso há uma cobrança adicional proporcional a...",parcial,sim,"D15-q02, D15-q03, D15-q05",D15-q03: [omitido: página gov.br (CC BY-ND 3.0); sha256 da33deef91c2; 400 caracteres]; D15-q05: [omitido: página gov.br (CC BY-ND 3.0); sha256 8519a121aa1e; 267 caracteres]
C2+M (oficial),"Não. A bandeira vermelha não é uma taxa a mais que você paga de forma separada. Ela indica que o custo de geração da energia está mais alto. Esse custo é adicionado à sua conta de energia, mas já ...",parcial,sim,"D15-q02, D15-q05, D15-q03, D15-q04, D12-s6-item37",D15-q03: [omitido: página gov.br (CC BY-ND 3.0); sha256 bfe1acf9bb8b; 400 caracteres]


## Erro 3 · T-007: a regra revogada

**Causa-raiz.** Até 2025, a distribuidora podia negar o ressarcimento quando o dano ocorria em
situação de emergência decretada (inciso VIII do art. 621 da REN 1.000 e alínea b do item 14 do
Módulo 9 do PRODIST). As duas regras foram revogadas pela REN 1.137/2025 (arts. 19 e 20), mas o
texto compilado ainda as mostra. O C2 original recebeu o art. 621 sem o inciso revogado, não
recebeu o ato da revogação e recusou por "falta de informação".

**Correção proposta.** Mostrar as notas de revogação e marcar os incisos revogados nos trechos
(correção D), com o alerta reforçado (E). A correção de fundo é na base: tirar do índice o texto
revogado, ou juntar a revogação ao próprio trecho, em vez de confiar que o modelo leia o alerta.

**Efeito medido.** A busca melhor piorou este caso. Com A+C, o reranker trouxe a alínea revogada
do PRODIST, e o modelo afirmou que a distribuidora **pode** negar, mesmo com o alerta de revogação
no trecho (incorreta). O C2+M responde "Não", mas apresenta a alínea revogada como exceção
válida (parcial, afirma algo falso). O sistema oficial não inclui a D; a limpeza da base não foi
feita.

In [5]:
mostrar("T-007")

**Pergunta (direta):** A distribuidora pode negar o ressarcimento de um aparelho queimado dizendo que a cidade estava em situação de emergência decretada?

**Gabarito:** Não. Essa possibilidade existia (inciso VIII do art. 621 da REN 1.000 e alínea b do item 14 do Módulo 9 do PRODIST), mas foi revogada pela REN 1.137/2025 (arts. 19 e 20); hoje a situação de emergência não está entre os casos em que a distribuidora pode deixar de ressarcir.

**Categoria:** recusa indevida (secundária: regra obsoleta) · gold entre os 3 trechos: sim · nota do portão: 0.6184

,resposta,veredito,afirma algo falso,trechos recebidos,fontes citadas
configuração,,,,,
C2 (original),"Falta informação sobre emergência decretada nos trechos. Procure a distribuidora, a ANEEL ou o Procon para esclarecer.",recusa,nao,"D01-art621-p1, D01-art621-p2, D01-art621-p3",D01-art621-p1: A distribuidora só pode eximir-se do dever de ressarcir no caso de: I - comprovar a inexistência de nexo causa
C2+A+C,"Sim, a distribuidora pode negar o ressarcimento se a perturbação que causou o dano tiver ocorrido em função de Situação de Emergência ou de Calamidade Pública decretada por autoridade competente. ...",incorreta,sim,"D08-s9.1-item14, D08-s9.3-item55-p1, D01-art612",D08-s9.1-item14: a perturbação que tenha dado causa ao dano reclamado tiver ocorrido em função de Situação de Emergência ou de
C2+M (oficial),"Não. A distribuidora não pode negar o ressarcimento apenas por dizer que a cidade estava em situação de emergência. O artigo 14 do Módulo 9 do PRODIST diz que, se a perturbação ocorrer por situaçã...",parcial,sim,"D08-s9.1-item14, D01-art612, D01-art621-p3, D01-art605, D08-s9.1-item21",D08-s9.1-item14: a perturbação que tenha dado causa ao dano reclamado tiver ocorrido em função de Situação de Emergência ou de


## Erro 4 · T-039: a resposta precisava de dois trechos

**Causa-raiz.** A pergunta pede dois prazos: o do resultado da análise e o do pagamento. Entre os
3 trechos veio o item 50 do PRODIST Módulo 9 (resultado em 15 dias), mas não o do pagamento (20
dias, art. 618 da REN 1.000 ou item 58 do PRODIST). O modelo respondeu a parte que tinha e disse,
corretamente, que o outro prazo não estava nos trechos.

**Correção proposta.** Busca híbrida com reranker (C) e mais trechos por pergunta (a M usa 5).

**Efeito medido.** Corrigido nas duas versões: com A+C vieram os arts. 617 e 618; com a M, os
itens 50 e 58. As duas dão 15 e 20 dias (correta).

In [6]:
mostrar("T-039")

**Pergunta (multi_trecho):** Pedi o ressarcimento um mês depois que meu aparelho queimou. Quanto tempo a empresa tem para me dar o resultado e, se aprovar, para pagar?

**Gabarito:** Resultado em até 15 dias (pedido feito em até 90 dias do dano) e, se aprovado, ressarcimento em até 20 dias.

**Categoria:** fragmentação · gold entre os 3 trechos: sim · nota do portão: 0.7097

,resposta,veredito,afirma algo falso,trechos recebidos,fontes citadas
configuração,,,,,
C2 (original),"A empresa tem até 15 dias após a verificação para dar o resultado, porque você pediu o ressarcimento um mês depois do dano. Se aprovado, o pagamento deve ser feito conforme o laudo e os documentos...",parcial,nao,"D14-q09, D08-s9.3-item50, D01-art602-p2",D14-q09: [omitido: página gov.br (CC BY-ND 3.0); sha256 816ef21539ed; 400 caracteres]; D08-s9.3-item50: 50. O prazo de Resposta para Solicitações de ressarcimento feitas em até 90 dias da data de ...
C2+A+C,"A empresa tem até 15 dias para te dar o resultado, porque você pediu o ressarcimento um mês depois que o aparelho queimou, o que está dentro dos 90 dias da ocorrência. Se aprovado, ela deve pagar ...",correta,nao,"D01-art617, D01-art618-p1, D01-art651",D01-art617: Art. 617. A distribuidora deve disponibilizar ao consumidor o resultado da análise da solicitação de ressarcim; D01-art618-p1: Art. 618. No caso de deferimento da solicitação de ressar...
C2+M (oficial),"A empresa tem até 15 dias para dar o resultado, porque você pediu o ressarcimento um mês depois que o aparelho queimou. Se aprovado, ela paga em até 20 dias a partir da resposta.",correta,nao,"D14-q09, D01-art617, D01-art618-p1, D08-s9.3-item50, D08-s9.3-item58","D08-s9.3-item50: 50. O prazo de Resposta para Solicitações de ressarcimento feitas em até 90 dias da data de provável ocorrênci; D08-s9.3-item58: 58. O prazo para o Ressarcimento é de 20 dias, con..."


## Erro 5 · T-011: a exceção apresentada como regra

**Causa-raiz.** O critério geral da Tarifa Social (renda mensal por pessoa de até meio salário
mínimo, Lei 12.212, art. 2º, I) não veio na busca. Veio o § 1º do mesmo artigo, que é o critério
excepcional (renda de até 3 salários mínimos, para família com pessoa que precisa de equipamentos
elétricos no tratamento), e o modelo o apresentou como a regra geral.

**Correção proposta.** Busca híbrida com reranker (C) e multiconsulta (M), para que a regra geral e
a exceção venham juntas.

**Efeito medido.** Corrigido nas duas versões: com o inciso I entre os trechos, as respostas dão o
meio salário mínimo como regra e os 3 salários mínimos como exceção (correta).

In [7]:
mostrar("T-011")

**Pergunta (direta):** Qual é o critério de renda do CadÚnico para ter direito à Tarifa Social de Energia Elétrica?

**Gabarito:** Família inscrita no CadÚnico com renda mensal por pessoa de até meio salário mínimo (ou ter na casa quem recebe o BPC).

**Categoria:** recuperação (secundária: citação não sustenta) · gold entre os 3 trechos: nao · nota do portão: 0.7085

,resposta,veredito,afirma algo falso,trechos recebidos,fontes citadas
configuração,,,,,
C2 (original),O critério de renda é ter renda mensal de até 3 (três) salários mínimos.,incorreta,sim,"D17-p2-q03, D02-art2-p2, D02-art3","D02-art2-p2: § 1º Excepcionalmente, será também beneficiada com a Tarifa Social de Energia Elétrica a unidade consumidora h"
C2+A+C,"O critério de renda é até meio salário mínimo nacional per capita. Isso se aplica a famílias inscritas no CadÚnico. Também há casos com renda até três salários mínimos, se houver membro com defici...",correta,nao,"D04-art1, D17-p2-q04, D02-art2-p1",D17-p2-q04: [omitido: página gov.br (CC BY-ND 3.0); sha256 38128fb77e18; 400 caracteres]; D02-art2-p1: I - seus moradores deverão pertencer a uma família inscrita no Cadastro Único para Programas ...
C2+M (oficial),"O critério de renda é que a família deve ter renda familiar mensal per capita de até meio salário mínimo nacional. Além disso, famílias com renda de até três salários mínimos podem ter direito se ...",correta,nao,"D04-art1, D17-p2-q04, D02-art2-p1, D17-p2-q03, D02-art2-p2",D02-art2-p1: I - seus moradores deverão pertencer a uma família inscrita no Cadastro Único para Programas Sociais do Govern; D04-art1: Será beneficiada com a Tarifa Social de Energia Elétrica - TS...


## Resumo e lições

In [8]:
resumo = pd.DataFrame(
    {nome: [f"{veredito(nome, i)['veredito']}" + (" (falso)" if veredito(nome, i)["falso"] == "sim" else "")
            for i in ESCOLHIDOS] for nome in execucoes},
    index=[f"{i} · {CATEGORIAS[erros[i]['categoria']]}" for i in ESCOLHIDOS],
)
resumo

,C2 (original),C2+A+C,C2+M (oficial)
T-021 · número alterado,parcial (falso),parcial (falso),parcial (falso)
T-028 · citação não sustenta,incorreta (falso),parcial (falso),parcial (falso)
T-007 · recusa indevida,recusa,incorreta (falso),parcial (falso)
T-039 · fragmentação,parcial,correta,correta
T-011 · recuperação,incorreta (falso),correta,correta


**Dois erros foram corrigidos pela busca** (T-039 e T-011). Quando o trecho certo chega ao modelo,
o qwen3 4B responde certo, inclusive separando regra e exceção.

**Três continuam errados, e são erros de interpretação** (T-021, T-028 e T-007). Em todos, o
número ou a regra que o modelo usa está em algum trecho, mas é de outra situação, contradiz o
ato normativo ou foi revogado. Por isso a verificação automática de números e de citações não os
detecta. Os mesmos padrões aparecem nos dois erros do C2+M no T3: a regra geral sem a exclusão da
classe residencial (T3-02) e os prazos de sistemas isolados no lugar do prazo comum (T3-17).

**Melhorar a busca pode piorar um caso.** No T-007, o reranker trouxe justamente o texto
revogado, e a resposta passou de recusa para afirmação falsa. Com uma base que guarda texto
revogado, um alerta no trecho não basta para um modelo de 4B.

**O que faria diferença** (não implementado):
1. identificar a situação do consumidor antes de escolher o prazo (T-021, T3-17);
2. tirar o texto revogado do índice ou juntá-lo à revogação no mesmo trecho (T-007);
3. dividir as páginas explicativas por situação e preferir o ato normativo quando as fontes
   divergem (T-021, T-028).